# 06 · Experimentos práticos  ⏱ 30 min

Agora é com vocês. Quatro experimentos curtos (meça, compare, conclua) e cinco exercícios com gabarito no final.

| # | Experimento | Pergunta |
|---|---|---|
| 1 | Pandas x Polars x DuckDB | Quem agrega 5 milhões de linhas mais rápido? |
| 2 | Paralelismo | Quanto ganho com mais núcleos? |
| 3 | CSV x Parquet | Quanto o formato muda tamanho e tempo? |
| 4 | Mais dados que memória | O que acontece com um limite de 300 MB? |

In [ ]:
import duckdb, time, os
import pandas as pd
import polars as pl

def cronometro(func, repeticoes=3):
    """Roda func algumas vezes e devolve o melhor tempo em segundos."""
    tempos = []
    for _ in range(repeticoes):
        t0 = time.perf_counter()
        func()
        tempos.append(time.perf_counter() - t0)
    return min(tempos)

con = duckdb.connect()

## Experimento 1 · Pandas x Polars x DuckDB

Mesma pergunta para as três ferramentas, lendo o mesmo Parquet: **receita e ticket médio por canal e mês, só vendas concluídas**.

In [ ]:
ARQ = "dados/vendas.parquet"

def com_pandas():
    df = pd.read_parquet(ARQ)
    df = df[df["status"] == "concluido"]
    df["receita"] = df["quantidade"] * df["preco_unitario"]
    df["mes"] = df["data_venda"].dt.to_period("M")
    return df.groupby(["canal", "mes"]).agg(receita=("receita", "sum"),
                                             pedidos=("pedido_id", "nunique")).reset_index()

def com_polars():
    return (pl.scan_parquet(ARQ)
              .filter(pl.col("status") == "concluido")
              .with_columns((pl.col("quantidade") * pl.col("preco_unitario")).alias("receita"),
                            pl.col("data_venda").dt.truncate("1mo").alias("mes"))
              .group_by("canal", "mes")
              .agg(pl.col("receita").sum(), pl.col("pedido_id").n_unique().alias("pedidos"))
              .collect())

def com_duckdb():
    return con.sql(f"""
        SELECT canal, date_trunc('month', data_venda) AS mes,
               sum(quantidade * preco_unitario) AS receita,
               count(DISTINCT pedido_id)        AS pedidos
        FROM '{ARQ}' WHERE status = 'concluido'
        GROUP BY ALL
    """).fetchall()

resultados = {nome: cronometro(f) for nome, f in
              [("pandas", com_pandas), ("polars", com_polars), ("duckdb", com_duckdb)]}
pd.Series(resultados, name="segundos").sort_values().to_frame()

In [ ]:
ax = pd.Series(resultados).sort_values().plot.barh(title="Tempo (s), menor é melhor", figsize=(7, 2.5))

**Discuta:** Polars e DuckDB costumam ficar próximos, e ambos bem à frente do pandas. Onde o pandas gasta tempo? (Dica: ele lê **todas** as colunas e materializa tudo em memória antes de filtrar. Polars em modo *lazy* e DuckDB leem só as colunas usadas.)

## Experimento 2 · Paralelismo

O DuckDB usa todos os núcleos por padrão. Vamos limitar com `SET threads`.

In [ ]:
consulta = f"""
    SELECT cliente_id, count(DISTINCT produto_id) AS produtos, sum(quantidade * preco_unitario) AS receita
    FROM '{ARQ}' GROUP BY ALL ORDER BY receita DESC LIMIT 10
"""
nucleos = os.cpu_count()
escala = {}
for n in sorted({1, 2, 4, nucleos}):
    if n > nucleos:
        continue
    con.sql(f"SET threads = {n}")
    escala[n] = cronometro(lambda: con.sql(consulta).fetchall())
con.sql("RESET threads")
pd.DataFrame({"segundos": escala, "aceleracao": {n: escala[1] / t for n, t in escala.items()}})

## Experimento 3 · CSV x Parquet

Mesmo dado (janeiro de 2025), dois formatos.

In [ ]:
csv, parquet = "dados/vendas_2025_01.csv", "dados/vendas_2025_01.parquet"
if not os.path.exists(parquet):
    con.sql(f"COPY (FROM '{csv}') TO '{parquet}'")

q = "SELECT canal, sum(quantidade * preco_unitario) FROM '{}' GROUP BY ALL"
pd.DataFrame({
    "tamanho_MB": {"CSV": os.path.getsize(csv) / 1e6, "Parquet": os.path.getsize(parquet) / 1e6},
    "consulta_ms": {"CSV": cronometro(lambda: con.sql(q.format(csv)).fetchall()) * 1000,
                    "Parquet": cronometro(lambda: con.sql(q.format(parquet)).fetchall()) * 1000},
}).round(1)

## Experimento 4 · Mais dados que memória

Limitamos o DuckDB a **300 MB** e pedimos um `ORDER BY` + `JOIN` em 5 milhões de linhas. O pandas precisaria de vários GB para isso. O DuckDB grava partes intermediárias em disco (*spilling*) e termina.

In [ ]:
con.sql("SET memory_limit = '300MB'")
con.sql("SET temp_directory = 'dados/tmp_spill'")

t0 = time.perf_counter()
con.sql(f"""
    COPY (
        SELECT v.*, c.cidade, c.uf, c.segmento
        FROM '{ARQ}' v JOIN 'dados/clientes.csv' c USING (cliente_id)
        ORDER BY c.uf, v.data_venda DESC
    ) TO 'dados/vendas_enriquecidas.parquet'
""")
print(f"Terminou em {time.perf_counter() - t0:.1f}s com limite de 300 MB")
con.sql("SELECT count(*) AS linhas FROM 'dados/vendas_enriquecidas.parquet'").show()
_ = con.sql("RESET memory_limit")

---
# Exercícios

Use `con.sql(...)` com os arquivos em `dados/`. O gabarito está no fim do notebook. Tente antes de olhar.

**1. (aquecimento)** Quais são as 5 cidades com mais clientes do segmento `empresa`?

In [ ]:
# sua resposta

**2. (friendly SQL)** Usando `PIVOT`, monte uma tabela com uma linha por `canal` e uma coluna por `status`, contando os itens vendidos em 2025.

In [ ]:
# sua resposta

**3. (janela)** Para cada mês de 2025, qual foi o **dia** de maior receita? Use `QUALIFY`.

In [ ]:
# sua resposta

**4. (logs)** Qual a porcentagem de requisições vindas de mobile (`contexto.mobile`) por rota? E a latência p95 de mobile x desktop?

In [ ]:
# sua resposta

**5. (desafio, 2.0)** Crie uma tabela `estoque(produto_id, quantidade)` com 100 unidades para os produtos 1 a 10, e uma tabela `movimentos`. Crie um **trigger** que, a cada `INSERT` em `movimentos(produto_id, qtd)`, desconte o estoque. Teste inserindo duas vendas.

In [ ]:
# sua resposta

---
# Gabarito

In [ ]:
# 1
con.sql("""
    SELECT cidade, uf, count(*) AS empresas
    FROM 'dados/clientes.csv' WHERE segmento = 'empresa'
    GROUP BY ALL ORDER BY empresas DESC LIMIT 5
""").show()

In [ ]:
# 2
con.sql("""
    PIVOT (FROM 'dados/vendas.parquet' WHERE year(data_venda) = 2025)
    ON status USING count(*)
    GROUP BY canal
    ORDER BY canal
""").show()

In [ ]:
# 3
con.sql("""
    SELECT date_trunc('month', data_venda)::DATE AS mes,
           data_venda::DATE                      AS dia,
           round(sum(quantidade * preco_unitario)) AS receita
    FROM 'dados/vendas.parquet'
    WHERE year(data_venda) = 2025 AND status = 'concluido'
    GROUP BY 1, 2
    QUALIFY row_number() OVER (PARTITION BY mes ORDER BY receita DESC) = 1
    ORDER BY mes
""").show()

In [ ]:
# 4
con.sql("""
    SELECT rota,
           round(100 * avg(contexto.mobile::INT), 1)                               AS pct_mobile,
           round(quantile_cont(latencia_ms, 0.95) FILTER (WHERE contexto.mobile))     AS p95_mobile,
           round(quantile_cont(latencia_ms, 0.95) FILTER (WHERE NOT contexto.mobile)) AS p95_desktop
    FROM 'dados/logs/*.jsonl'
    GROUP BY ALL ORDER BY rota
""").show()

In [ ]:
# 5
con.sql("""
    CREATE OR REPLACE TABLE estoque AS SELECT range AS produto_id, 100 AS quantidade FROM range(1, 11);
    CREATE OR REPLACE TABLE movimentos (produto_id INTEGER, qtd INTEGER);

    CREATE TRIGGER baixa_estoque
    AFTER INSERT ON movimentos
    REFERENCING NEW TABLE AS novos
    FOR EACH STATEMENT
    UPDATE estoque SET quantidade = estoque.quantidade - n.total
    FROM (SELECT produto_id, sum(qtd) AS total FROM novos GROUP BY produto_id) n
    WHERE estoque.produto_id = n.produto_id;
""")
con.sql("INSERT INTO movimentos VALUES (1, 3), (2, 5)")
con.sql("INSERT INTO movimentos VALUES (1, 10)")
con.sql("FROM estoque WHERE produto_id <= 3 ORDER BY produto_id").show()

In [ ]:
con.close()